## load listing dataset

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "property_listings_raw.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Shape:", df.shape)
display(df.head(3))

Shape: (6495, 57)


,id,crm_id,strapi_id,slug,title,listing_type,department,search_type,status,price,...,agent_phone,agent_whatsapp,agent_designation,agent_languages,agent_photo_url,agent_slug,video_tour,virtual_tour,enriched_at,property_type
0,9971,DBI215069,9971,2-bedroom-apartment-for-sale-in-mosela-watersi...,2 bedroom apartment for sale,purchase,residential,sales,For Sale,2850000.0,...,+971 58 577 0306,+971 58 577 0306,Greens & Views Consultant,"[""English"",""Turkish""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,murat-topcuoglu,[],[],2026-09-17T17:03:40Z,apartment
1,12292,DUB203155,12292,3-bedroom-villa-for-sale-in-district-8-jumeira...,3 bedroom villa for sale,purchase,residential,sales,For Sale,7195000.0,...,+971 50 598 0269,+971 50 598 0269,Associate Partner,"[""English""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,cathal-o-neill,[],[],2026-09-17T18:04:47Z,villa
2,10051,DBI210323,10051,2-bedroom-apartment-for-sale-in-al-ghozlan-1-t...,2 bedroom apartment for sale,purchase,residential,sales,For Sale,2250000.0,...,+971 58 577 0306,+971 58 577 0306,Greens & Views Consultant,"[""English"",""Turkish""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,murat-topcuoglu,[],[],2026-09-17T18:04:47Z,apartment


## column names and types

In [2]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing": [int(df[c].isna().sum()) for c in df.columns],
    "unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
})

display(schema)


,column,dtype,missing,unique
0,id,int64,0,6495
1,crm_id,str,0,6495
2,strapi_id,int64,0,6495
3,slug,str,0,4128
4,title,str,0,56
5,listing_type,str,0,2
6,department,str,0,2
7,search_type,str,0,2
8,status,str,0,4
9,price,float64,0,1100


## find likely filter columns

In [3]:
def find_column(candidates, columns):
    """Find a column using case-insensitive exact name matching."""
    normalized = {
        str(c).strip().lower().replace("-", "_").replace(" ", "_"): c
        for c in columns
    }

    for candidate in candidates:
        key = candidate.strip().lower().replace("-", "_").replace(" ", "_")
        if key in normalized:
            return normalized[key]

    return None


ALIASES = {
    "price": ["price", "listing_price", "asking_price", "amount"],
    "bedrooms": ["bedrooms", "beds", "bedroom", "num_bedrooms"],
    "bathrooms": ["bathrooms", "baths", "num_bathrooms"],
    "community": ["community", "area", "neighborhood", "location"],
    "property_type": ["property_type", "type", "property_category"],
    "listing_type": ["listing_type", "purpose", "offering_type", "transaction_type"],
    "description": ["description", "property_description", "summary", "title"],
    "property_id": ["property_id", "listing_id", "id", "reference"],
    "url": ["url", "listing_url", "property_url", "link"],
}

COLS = {
    key: find_column(aliases, df.columns)
    for key, aliases in ALIASES.items()
}

display(pd.DataFrame(
    [{"field": k, "detected_column": v} for k, v in COLS.items()]
))

,field,detected_column
0,price,price
1,bedrooms,beds
2,bathrooms,bathrooms
3,community,community
4,property_type,property_type
5,listing_type,listing_type
6,description,description
7,property_id,id
8,url,link


In [ ]:
import pandas as pd

work = df.copy()

# Normalize fields used by hard filters.
work["_price"] = pd.to_numeric(work["price"], errors="coerce")
work["_bedrooms"] = pd.to_numeric(work["beds"], errors="coerce")
work["_bathrooms"] = pd.to_numeric(work["bathrooms"], errors="coerce")

work["_community"] = work["community"].fillna("").astype(str)
work["_property_type"] = work["property_type"].fillna("").astype(str)
work["_listing_type"] = work["listing_type"].fillna("").astype(str)

def filter_properties(
    data,
    max_price=None,
    min_price=None,
    bedrooms=None,
    min_bedrooms=None,
    min_bathrooms=None,
    community_contains=None,
    property_type=None,
    listing_type=None,
):
    result = data.copy()

    # Exclude listings with missing or non-positive prices
    result = result[result["_price"].notna() & (result["_price"] > 0)]

    if max_price is not None:
        result = result[result["_price"] <= max_price]

    if min_price is not None:
        result = result[result["_price"] >= min_price]

    if bedrooms is not None:
        result = result[result["_bedrooms"] == bedrooms]

    if min_bedrooms is not None:
        result = result[result["_bedrooms"] >= min_bedrooms]

    if min_bathrooms is not None:
        result = result[result["_bathrooms"] >= min_bathrooms]

    if community_contains:
        result = result[
            result["_community"].str.contains(
                community_contains, case=False, regex=False
            )
        ]

    if property_type:
        result = result[
            result["_property_type"].str.contains(
                property_type, case=False, regex=False
            )
        ]

    if listing_type:
        result = result[
            result["_listing_type"].str.casefold()
            == listing_type.casefold()
        ]

    return result.sort_values("_price", ascending=True)


In [5]:
results = filter_properties(
    work,
    max_price=2_000_000,
    bedrooms=2,
    community_contains="Dubai Marina",
    listing_type="purchase",
)

print("Matching properties:", len(results))

display(
    results[
        [
            "id",
            "title",
            "price",
            "beds",
            "bathrooms",
            "community",
            "property_type",
            "listing_type",
            "link",
        ]
    ].head(10)
)

Matching properties: 6


,id,title,price,beds,bathrooms,community,property_type,listing_type,link
577,151291,2 bedroom apartment for sale,1350000.0,2,2,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...
585,151408,2 bedroom apartment for sale,1395000.0,2,2,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...
1229,158404,2 bedroom apartment for sale,1500000.0,2,2,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...
517,150053,2 bedroom apartment for sale,1735000.0,2,2,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...
657,152746,2 bedroom apartment for sale,1875000.0,2,3,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...
740,154107,2 bedroom apartment for sale,2000000.0,2,3,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...


In [6]:
print("Listing types:")
display(work["listing_type"].value_counts(dropna=False))

print("Search types:")
display(work["search_type"].value_counts(dropna=False))

print("Property types:")
display(work["property_type"].value_counts(dropna=False))

print("Communities:")
display(
    work["community"]
    .value_counts(dropna=False)
    .head(20)
)

Listing types:


listing_type
rental      4973
purchase    1522
Name: count, dtype: int64

Search types:


search_type
lettings    4973
sales       1522
Name: count, dtype: int64

Property types:


property_type
apartment         3904
villa             1717
townhouse          556
office space       252
retail              37
NaN                 18
land                 5
warehouse            4
whole building       1
labour camp          1
Name: count, dtype: int64

Communities:


community
Dubai Marina                 551
Dubai Hills Estate           423
Downtown Dubai               405
Business Bay                 360
Palm Jumeirah                318
Arabian Ranches              246
Jumeirah Village Circle      239
Mohammed Bin Rashid City     220
Dubai Creek Harbour          211
The Springs                  207
Jumeirah Lake Towers         199
The Views                    188
The Greens                   126
Al Furjan                    118
Jumeirah Park                110
Jumeirah Golf Estates        109
Reem                         100
Jumeirah Beach Residence      98
Jumeirah Village Triangle     93
Tilal Al Ghaf                 91
Name: count, dtype: int64